# NDC Search

In [1]:
import pandas as pd
import requests
from pandas import json_normalize


# 1. Define the API Endpoint

# Use 11 digit NDC
ndc = '00002147180'

# url = "http://localhost:4000/REST/ndcproperties.json?id=00002147180"
url = f"http://localhost:4000/REST/ndcproperties.json?id={ndc}"

# 2. Make the Request
response = requests.get(url)

# 3. Check if the request was successful (Status Code 200)
if response.status_code == 200:
    # Convert the JSON response into a Python object (usually a list of dicts)
    json_data = response.json()
    
    # 4. Load into Pandas
    # df = pd.DataFrame(json_data)
    
    # Flattens nested JSON (e.g., separates 'address.street', 'address.city')
    df = pd.json_normalize(json_data)
    
    # Display the first few rows
    display(df.head())
else:
    print(f"Error: Failed to retrieve data. Status code: {response.status_code}")

,ndcPropertyList.ndcProperty
0,"[{'ndcItem': '00002147180', 'ndc9': '0002-1471..."


In [4]:
response = json_data["ndcPropertyList"]["ndcProperty"][0]

# display(response)

## Obtain rxcui

In [5]:
rxcui = response["rxcui"]
ndc10 = response["ndc10"]
marketing = response["propertyConceptList"]["propertyConcept"][4]["propValue"]
labeler =  response["propertyConceptList"]["propertyConcept"][0]["propValue"]

print(f"Labeler: {labeler}")
print(f"rxcui: {rxcui}")
print(f"NDC 10: {ndc10}")
print(f"Marketing Status: {marketing}")

Labeler: Eli Lilly and Company
rxcui: 2601770
NDC 10: 0002-1471-80
Marketing Status: ACTIVE


## Find Active Ingredients

In [22]:
active_products_url = f"http://localhost:4000/REST/rxcui/{rxcui}/active.json"

response = requests.get(active_products_url)

# 3. Check if the request was successful (Status Code 200)
if response.status_code == 200:
    # Convert the JSON response into a Python object (usually a list of dicts)
    json_data = response.json()
    
    # 4. Load into Pandas
    # df = pd.DataFrame(json_data)
    
    # Flattens nested JSON (e.g., separates 'address.street', 'address.city')
    df = pd.json_normalize(json_data)
    
    # Display the first few rows
    display(df.head())
else:
    print(f"Error: Failed to retrieve data. Status code: {response.status_code}")

,minConceptGroup.minConcept
0,"[{'rxcui': '2601770', 'name': '0.5 ML tirzepat..."


In [29]:
rxcui = json_data["minConceptGroup"]["minConcept"][0]["rxcui"]
name = json_data["minConceptGroup"]["minConcept"][0]["name"]

print(f"rxcui: {rxcui}")
# print("\n")
print(f"Name: {name}")

rxcui: 2601770
Name: 0.5 ML tirzepatide 20 MG/ML Auto-Injector [Mounjaro]


## Get all Properties

In [33]:
prop_url = f"http://localhost:4000/REST/rxcui/{rxcui}/allProperties.json?prop=names+codes"

# 2. Make the Request
response = requests.get(prop_url)

# 3. Check if the request was successful (Status Code 200)
if response.status_code == 200:
    # Convert the JSON response into a Python object (usually a list of dicts)
    json_data = response.json()
    
    # 4. Load into Pandas
    # df = pd.DataFrame(json_data)
    
    # Flattens nested JSON (e.g., separates 'address.street', 'address.city')
    df = pd.json_normalize(json_data)
    
    # Display the first few rows
    display(df.head())
else:
    print(f"Error: Failed to retrieve data. Status code: {response.status_code}")

,propConceptGroup.propConcept
0,"[{'propCategory': 'CODES', 'propName': 'MMSL_C..."


In [34]:
# 1. Create the base dataframe
prop_list = json_data.get('propConceptGroup', {}).get('propConcept', [])
df = pd.DataFrame(prop_list)

# 2. Pivot the data
# We Group By 'propName' and apply a function to join duplicates
df_wide = df.groupby('propName')['propValue'].apply(lambda x: '; '.join(x)).to_frame().T

# 3. (Optional) Add the RxCUI from your loop variable so you know which drug this is
df_wide.insert(0, 'rxcui', '352125') # Replace '352125' with your variable

display(df_wide)

propName,rxcui,MMSL_CODE,NDA,Prescribable Synonym,RXNAV_STR,RxCUI,RxNorm Name,RxNorm Synonym,SPL_SET_ID
propValue,352125,BD38737,NDA215866,mounjaro 10 MG in 0.5 ML Auto-Injector,0.5 ML Mounjaro 20 MG/ML Auto-Injector,2601770,0.5 ML tirzepatide 20 MG/ML Auto-Injector [Mou...,0.5 ML Mounjaro 20 MG/ML Auto-Injector; Mounja...,d2d7da5d-ad07-4228-955f-cf7e355c8cc0


In [35]:
# 1. Target the specific list inside the JSON
# We use .get() to avoid errors if the key is missing
prop_list = json_data.get('propConceptGroup', {}).get('propConcept', [])

# 2. Load directly into DataFrame
df = pd.DataFrame(prop_list)

# Optional: filter to just what you want
# df = df[df['propCategory'] == 'CODES']

display(df)

,propCategory,propName,propValue
0,CODES,MMSL_CODE,BD38737
1,CODES,NDA,NDA215866
2,CODES,RxCUI,2601770
3,CODES,SPL_SET_ID,d2d7da5d-ad07-4228-955f-cf7e355c8cc0
4,NAMES,Prescribable Synonym,mounjaro 10 MG in 0.5 ML Auto-Injector
5,NAMES,RXNAV_STR,0.5 ML Mounjaro 20 MG/ML Auto-Injector
6,NAMES,RxNorm Name,0.5 ML tirzepatide 20 MG/ML Auto-Injector [Mou...
7,NAMES,RxNorm Synonym,0.5 ML Mounjaro 20 MG/ML Auto-Injector
8,NAMES,RxNorm Synonym,Mounjaro 10 MG per 0.5 ML Auto-Injector
